## HW2 Submission Template

Follow this template for both questions of HW2, filling in your code in the cells indicated. Feel free to add as many cells as you want, but maintain the same layout. Remember that each question also requires a report in a PDF file (to be submitted together). 

### Question 1

#### Load AirBNB data
Run the two cells below to read the airbnb data.
See [this notebook](https://adb-2068980679427394.14.azuredatabricks.net/editor/notebooks/2178897309767401?o=2068980679427394) for more details.

In [ ]:
# config
storage_account = "lab94290"  
container = "airbnb"

sas_token="sp=rle&st=2025-12-05T10:01:31Z&se=2026-02-26T18:23:31Z&spr=https&sv=2024-11-04&sr=c&sig=zdjdrz3xY%2BYfIMQt8Dvg2tzzKMHNR4sXttO%2BTUhAvJM%3D"
sas_token = sas_token.lstrip('?')
spark.conf.set(f"fs.azure.account.auth.type.{storage_account}.dfs.core.windows.net", "SAS")
spark.conf.set(f"fs.azure.sas.token.provider.type.{storage_account}.dfs.core.windows.net", "org.apache.hadoop.fs.azurebfs.sas.FixedSASTokenProvider")
spark.conf.set(f"fs.azure.sas.fixed.token.{storage_account}.dfs.core.windows.net", sas_token)

In [ ]:
path = f"abfss://{container}@{storage_account}.dfs.core.windows.net/airbnb_1_12_parquet"
airbnb_df = spark.read.parquet(path)

In [ ]:
display(airbnb_df)

#### Your Solution

In [ ]:
from pyspark.sql.functions import count, col

# We will want currency uniformity ==> USD
airbnb_df.groupBy("currency").count().show() 
airbnb_df = airbnb_df.fillna({"currency": "USD"})
airbnb_df = airbnb_df.filter(col("currency") == "USD") 

# To check if 'discount' attribute is relevant for us.

airbnb_df.select(count(col("discount")).alias("discount_non_null")).show() 


In [ ]:
from pyspark.sql import functions as F
from pyspark.sql.functions import col, when, lit

# ============================================================
# 1. Filter out listings with missing or zero price
# ============================================================
airbnb_df = airbnb_df.filter(
    col("price").isNotNull() & (col("price") != 0)
)

# ============================================================
# 2. Cast "guests" to integer
#    We create an integer version of the guests column.
# ============================================================
airbnb_df = airbnb_df.withColumn(
    "guests_int",
    col("guests").cast("int")
)

# ============================================================
# 3. Compute median number of guests (over non-null rows)
#    This is used as a fallback for missing/invalid guest counts.
# ============================================================
median_guests = airbnb_df.approxQuantile("guests_int", [0.5], 0.05)[0]

# ============================================================
# 4. Define a safe guests column:
#    - If guests_int is NULL or <= 0 → use median_guests
#    - Otherwise use guests_int as is
# ============================================================
airbnb_df = airbnb_df.withColumn(
    "guests_safe",
    when(col("guests_int").isNull() | (col("guests_int") <= 0), int(median_guests))
    .otherwise(col("guests_int"))
)

# ============================================================
# 5. Compute price per guest
#    Here we use the *nightly* price.
#    If you prefer total trip price, replace "price" with "total_price".
# ============================================================
airbnb_df = airbnb_df.withColumn(
    "price_per_guest",
    (col("price") / col("guests_safe")).cast("double")
)

# ============================================================
# 6. Robust normalization of price_per_guest using quantiles
#    Instead of min–max over all data (which is very sensitive to outliers),
#    we use the 5th and 95th percentiles as effective bounds.
# ============================================================

# Compute 5th and 95th percentiles
ppg_low, ppg_high = airbnb_df.approxQuantile(
    "price_per_guest",
    [0.05, 0.95],
    0.01
)

# Protect against degenerate case
if ppg_high == ppg_low:
    ppg_high = ppg_low + 1.0

# Clip price_per_guest to [ppg_low, ppg_high]
airbnb_df = airbnb_df.withColumn(
    "price_per_guest_clipped",
    when(col("price_per_guest") < ppg_low, ppg_low)
    .when(col("price_per_guest") > ppg_high, ppg_high)
    .otherwise(col("price_per_guest"))
)

# Normalize clipped price_per_guest to [0, 1]
ppg_range = ppg_high - ppg_low

airbnb_df = airbnb_df.withColumn(
    "price_per_guest_norm",
    (col("price_per_guest_clipped") - lit(ppg_low)) / lit(ppg_range)
)

# Clamp to [0, 1] to avoid edge cases
airbnb_df = airbnb_df.withColumn(
    "price_per_guest_norm",
    when(col("price_per_guest_norm") < 0, 0.0)
    .when(col("price_per_guest_norm") > 1, 1.0)
    .otherwise(col("price_per_guest_norm"))
)

# ============================================================
# 7. Convert to "price_score":
#    - Cheaper per guest → score close to 1
#    - More expensive per guest → score close to 0
#    Here we use a mild non-linearity (**1.1) to slightly punish
#    expensive listings more strongly.
# ============================================================
airbnb_df = airbnb_df.withColumn(
    "price_score",
    (1.0 - col("price_per_guest_norm")) ** 1.1
)

display(airbnb_df)


In [ ]:
# Distribution Detection

airbnb_df.select("ratings").describe().show()
airbnb_df.select("property_number_of_reviews").describe().show()

airbnb_df.groupBy("ratings").count().orderBy("ratings").show()
airbnb_df.groupBy("property_number_of_reviews").count().orderBy("property_number_of_reviews").show()


In [ ]:
from pyspark.sql.functions import when, col, log1p, lit

# Ratings Processing 

airbnb_df = airbnb_df.filter(
    col("ratings").isNotNull() & (col("ratings") != 0)
)
airbnb_df = airbnb_df.withColumn("ratings_int", col("ratings").cast("int")) 

rating_min = airbnb_df.agg({"ratings_int": "min"}).first()[0]
rating_max = airbnb_df.agg({"ratings_int": "max"}).first()[0]

airbnb_df = airbnb_df.withColumn( 
    "normalized_ratings",
    (col("ratings_int") - rating_min) / (rating_max - rating_min)
)

# Reviews Processing 

airbnb_df = airbnb_df.withColumn("property_number_of_reviews_int", col("property_number_of_reviews").cast("int")) 
median_reviews = airbnb_df.approxQuantile("property_number_of_reviews_int", [0.5], 0.01)[0]

airbnb_df = airbnb_df.withColumn(
    "property_number_of_reviews_int",
    when(
        col("property_number_of_reviews_int").isNull(), 
        lit(median_reviews)
    ).when(
        col("property_number_of_reviews_int") == 0,
        lit(median_reviews)
    ).otherwise(col("property_number_of_reviews_int"))
)

airbnb_df = airbnb_df.withColumn(
    "reviews_log",
    log1p(col("property_number_of_reviews_int")) 
)

reviews_min = airbnb_df.agg({"reviews_log": "min"}).first()[0]
reviews_max = airbnb_df.agg({"reviews_log": "max"}).first()[0]

airbnb_df = airbnb_df.withColumn(
    "normalized_reviews",
    (col("reviews_log") - reviews_min) / (reviews_max - reviews_min) 
)

airbnb_df = airbnb_df.withColumn(
    "z_rating",
    2 * col("normalized_ratings") - lit(1.0)
)

airbnb_df = airbnb_df.withColumn(
    "trust_factor",
    0.5 + 0.5 * col("normalized_reviews")
)

airbnb_df = airbnb_df.withColumn(
    "z_quality",
    col("z_rating") * col("trust_factor")
)

airbnb_df = airbnb_df.withColumn(
    "propery_rating_reviews_score",
    (col("z_quality") + lit(1.0)) / lit(2.0)
)


In [ ]:
# Distribution Detection 

airbnb_df.select("host_rating").describe().show()
airbnb_df.select("host_number_of_reviews").describe().show()

airbnb_df.groupBy("host_rating").count().orderBy("host_rating").show()
airbnb_df.groupBy("host_number_of_reviews").count().orderBy("host_number_of_reviews").show()


In [ ]:
from pyspark.sql.functions import when, col, log1p, lit 

# Host_Ratings Processing 

airbnb_df = airbnb_df.filter(
    col("host_rating").isNotNull() & (col("host_rating") != 0)
)
airbnb_df = airbnb_df.withColumn("host_rating_int", col("host_rating").cast("int")) 

rating_min = airbnb_df.agg({"host_rating_int": "min"}).first()[0]
rating_max = airbnb_df.agg({"host_rating_int": "max"}).first()[0]

airbnb_df = airbnb_df.withColumn( 
    "normalized_host_rating",
    (col("host_rating_int") - rating_min) / (rating_max - rating_min)
)

# Host_Reviews Processing 

airbnb_df = airbnb_df.withColumn("host_number_of_reviews_int", col("host_number_of_reviews").cast("int")) 
median_reviews = airbnb_df.approxQuantile("host_number_of_reviews_int", [0.5], 0.01)[0]

airbnb_df = airbnb_df.withColumn(
    "host_number_of_reviews_int",
    when(
        col("host_number_of_reviews_int").isNull(), 
        lit(median_reviews)
    ).when(
        col("host_number_of_reviews_int") == 0,
        lit(median_reviews)
    ).otherwise(col("host_number_of_reviews_int")) 
)

airbnb_df = airbnb_df.withColumn(
    "host_reviews_log",
    log1p(col("host_number_of_reviews_int")) 
)

reviews_min = airbnb_df.agg({"host_reviews_log": "min"}).first()[0]
reviews_max = airbnb_df.agg({"host_reviews_log": "max"}).first()[0]

airbnb_df = airbnb_df.withColumn(
    "normalized_host_reviews",
    (col("host_reviews_log") - reviews_min) / (reviews_max - reviews_min)
)

airbnb_df = airbnb_df.withColumn(
    "z_host_rating",
    2 * col("normalized_host_rating") - lit(1.0)
)

airbnb_df = airbnb_df.withColumn(
    "host_trust_factor",
    0.5 + 0.5 * col("normalized_host_reviews")
)

airbnb_df = airbnb_df.withColumn(
    "z_host_quality",
    col("z_host_rating") * col("host_trust_factor")
)

airbnb_df = airbnb_df.withColumn(
    "host_rating_reviews_score",
    (col("z_host_quality") + lit(1.0)) / lit(2.0)
)

display(airbnb_df)

In [ ]:
from pyspark.sql.types import ArrayType, StructType, StructField, StringType
from pyspark.sql.functions import (
    col,
    from_json,
    expr,
    array_contains,
    lit,
    greatest
)

# ============================================================
# 1. Define schema for the amenities JSON array
# ============================================================
amenities_schema = ArrayType(
    StructType([
        StructField("group_name", StringType()),
        StructField("items", ArrayType(
            StructType([
                StructField("name", StringType()),
                StructField("value", StringType())
            ])
        ))
    ])
)

# ============================================================
# 2. Parse the raw amenities JSON into a structured column
# ============================================================
airbnb_df = airbnb_df.withColumn(
    "amenities_parsed",
    from_json(col("amenities"), amenities_schema)
)

# ============================================================
# 3. Extract all amenity values and names into flattened arrays
#     - amenity_values: machine-readable codes like SYSTEM_XXXX
#     - amenity_names: user-facing textual labels
# ============================================================
airbnb_df = airbnb_df.withColumn(
    "amenity_values",
    expr("flatten(transform(amenities_parsed, g -> transform(g.items, i -> i.value)))")
)

airbnb_df = airbnb_df.withColumn(
    "amenity_names",
    expr("flatten(transform(amenities_parsed, g -> transform(g.items, i -> i.name)))")
)

# ============================================================
# 4. Heating / Cooling Features
#     We separate AC and Heating, then combine them into a
#     temperature comfort score (temp_score).
#     Logic:
#       - If either AC or Heating exists → base 2.5 points
#       - If BOTH exist → additional +0.5 points
# ============================================================
airbnb_df = airbnb_df.withColumn(
    "has_ac",
    array_contains(col("amenity_values"), "SYSTEM_SNOWFLAKE").cast("int")
)

airbnb_df = airbnb_df.withColumn(
    "has_heating",
    array_contains(col("amenity_values"), "SYSTEM_THERMOMETER").cast("int")
)

airbnb_df = airbnb_df.withColumn(
    "temp_score",
    1.5 * greatest(col("has_ac"), col("has_heating")) +
    0.5 * ((col("has_ac") + col("has_heating") == 2).cast("int"))
)

# ============================================================
# 5. Internet / Connectivity Features
# ============================================================
airbnb_df = airbnb_df.withColumn(
    "has_wifi",
    (
        array_contains(col("amenity_values"), "SYSTEM_WI_FI") |
        array_contains(col("amenity_values"), "SYSTEM_PORTABLE_WI_FI")
    ).cast("int")
)

# ============================================================
# 6. Parking Features
#     IMPORTANT: The value SYSTEM_MAPS_CAR_RENTAL appears for ALL
#     parking types. Therefore we MUST rely on the textual "name".
#
#     - Free parking:  "Free parking on premises", "Free street parking", etc.
#     - Paid parking:  "Paid parking on/off premises", "Paid parking lot", etc.
# ============================================================
airbnb_df = airbnb_df.withColumn(
    "has_parking_free",
    expr("exists(amenity_names, n -> lower(n) LIKE '%free%park%')").cast("int")
)

airbnb_df = airbnb_df.withColumn(
    "has_parking_paid",
    expr("exists(amenity_names, n -> lower(n) LIKE '%paid%park%')").cast("int")
)

# ============================================================
# 7. Pool / Jacuzzi Features
# ============================================================
airbnb_df = airbnb_df.withColumn(
    "has_pool",
    array_contains(col("amenity_values"), "SYSTEM_POOL").cast("int")
)

airbnb_df = airbnb_df.withColumn(
    "has_hot_tub",
    array_contains(col("amenity_values"), "SYSTEM_JACUZZI").cast("int")
)

# ============================================================
# 8. Family-Friendly Features
# ============================================================
airbnb_df = airbnb_df.withColumn(
    "has_family",
    (
        array_contains(col("amenity_values"), "SYSTEM_CRIB") |
        array_contains(col("amenity_values"), "SYSTEM_PACK_N_PLAY") |
        array_contains(col("amenity_values"), "SYSTEM_HIGH_CHAIR") |
        array_contains(col("amenity_values"), "SYSTEM_TOYS") |
        array_contains(col("amenity_values"), "SYSTEM_BOARD_GAMES")
    ).cast("int")
)

# ============================================================
# 9. Nature Views (ocean, mountain, garden, beach)
# ============================================================
airbnb_df = airbnb_df.withColumn(
    "has_nature_view",
    (
        array_contains(col("amenity_values"), "SYSTEM_VIEW_OCEAN") |
        array_contains(col("amenity_values"), "SYSTEM_VIEW_MOUNTAIN") |
        array_contains(col("amenity_values"), "SYSTEM_BEACH") |
        array_contains(col("amenity_values"), "SYSTEM_FLOWER")
    ).cast("int")
)

# ============================================================
# 10. City View
# ============================================================
airbnb_df = airbnb_df.withColumn(
    "has_city_view",
    array_contains(col("amenity_values"), "SYSTEM_VIEW_CITY").cast("int")
)

# ============================================================
# 11. Kitchen Detection
#     We use name-based detection since not all kitchens have SYSTEM tags.
# ============================================================
airbnb_df = airbnb_df.withColumn(
    "has_kitchen",
    (
        expr("exists(amenity_names, n -> n LIKE 'Kitchen%')") |
        expr("exists(amenity_names, n -> n LIKE 'Kitchenette%')")
    ).cast("int")
)

# ============================================================
# 12. Workspace
# ============================================================
airbnb_df = airbnb_df.withColumn(
    "has_workspace",
    (
        array_contains(col("amenity_values"), "SYSTEM_WORKSPACE") |
        expr("exists(amenity_names, n -> lower(n) LIKE '%workspace%')")
    ).cast("int")
)

# ============================================================
# 13. Laundry Features (Washer / Dryer)
# ============================================================
airbnb_df = airbnb_df.withColumn(
    "has_washer",
    array_contains(col("amenity_values"), "SYSTEM_WASHER").cast("int")
)

airbnb_df = airbnb_df.withColumn(
    "has_dryer",
    array_contains(col("amenity_values"), "SYSTEM_DRYER").cast("int")
)

# ============================================================
# 14. Final Amenities Score (Raw)
#
#     Combines all features with predefined weights.
#     temp_score already contributes up to 3.0 points.
# ============================================================
airbnb_df = airbnb_df.withColumn(
    "amenities_score_raw",
    col("temp_score") +                     # Temperature comfort score (0–2.0)
    2 * col("has_wifi") +
    2.4 * col("has_parking_free") +         # Higher value than paid parking
    0.3 * col("has_parking_paid") +
    2.5 * col("has_pool") +
    1.5 * col("has_hot_tub") +
    1.2 * col("has_family") +
    1.8 * col("has_nature_view") +
    1.1 * col("has_city_view") +
    1.8 * col("has_kitchen") +
    0.9 * col("has_workspace") +
    0.7 * col("has_washer") +
    0.5 * col("has_dryer")
)

# ============================================================
# 15. Normalize Amenities Score to range [0, 1]
#     max_amenities_score = maximum theoretical score
# ============================================================
max_amenities_score = 18.7 # Sum of all weights

airbnb_df = airbnb_df.withColumn(
    "amenities_score",
    col("amenities_score_raw") / lit(max_amenities_score)
)


In [ ]:
from pyspark.sql.types import ArrayType, StructType, StructField, StringType
from pyspark.sql.functions import (
    col,
    from_json,
    expr,
    when,
    size,
    lit
)

# ============================================================
# 1. Define schema for cancellation_policy JSON
#     Each entry contains:
#       - cancellation_name  (e.g., "Full refund before…")
#       - cancellation_value (unused here but kept for completeness)
# ============================================================
cancellation_schema = ArrayType(
    StructType([
        StructField("cancellation_name", StringType()),
        StructField("cancellation_value", StringType())
    ])
)

# ============================================================
# 2. Parse the cancellation_policy JSON string
# ============================================================
airbnb_df = airbnb_df.withColumn(
    "cancellation_parsed",
    from_json(col("cancellation_policy"), cancellation_schema)
)

# ============================================================
# 3. Replace NULL with an empty array
#     This prevents errors when applying array operations
#     and defines a clear semantic:
#       NULL → treated as "no cancellation info provided"
# ============================================================
airbnb_df = airbnb_df.withColumn(
    "cancellation_parsed",
    when(col("cancellation_parsed").isNull(), expr("array()"))
    .otherwise(col("cancellation_parsed"))
)

# ============================================================
# 4. Extract only the cancellation policy names
#     Example output:
#       ["Full refund before…", "Partial refund until…"]
# ============================================================
airbnb_df = airbnb_df.withColumn(
    "cancellation_names",
    expr("transform(cancellation_parsed, c -> c.cancellation_name)")
)

# ============================================================
# 5. Detect refund types
#     We classify each listing into one of:
#       - full refund
#       - partial refund
#       - no refund
#
#     Important edge-case behavior:
#       - If there is *no* policy data → treat as "no refund"
# ============================================================

# -----------------------------
# Full refund detection
# -----------------------------
airbnb_df = airbnb_df.withColumn(
    "has_full_refund",
    when(
        size(col("cancellation_parsed")) == 0,        # No data → not full refund
        lit(False)
    ).otherwise(
        expr("exists(cancellation_names, n -> n LIKE 'Full refund%')")
    ).cast("int")
)

# -----------------------------
# Partial refund detection
# -----------------------------
airbnb_df = airbnb_df.withColumn(
    "has_partial_refund",
    when(
        size(col("cancellation_parsed")) == 0,        # No data → not partial refund
        lit(False)
    ).otherwise(
        expr("exists(cancellation_names, n -> n LIKE 'Partial refund%')")
    ).cast("int")
)

# -----------------------------
# No refund detection
#     Logic:
#       - If array is empty → assume no refund
#       - Else → check explicitly
# -----------------------------
airbnb_df = airbnb_df.withColumn(
    "has_no_refund",
    when(
        size(col("cancellation_parsed")) == 0,        # No data → assume no refund
        lit(True)
    ).otherwise(
        expr("exists(cancellation_names, n -> n LIKE 'No refund%')")
    ).cast("int")
)

# ============================================================
# 6. Construct cancellation_score
#
#     Score interpretation:
#       1.0 → full refund
#       0.5 → partial refund
#       0.0 → no refund
# ============================================================
airbnb_df = airbnb_df.withColumn(
    "cancellation_score",
    when(col("has_full_refund") == 1, 1.0)
    .when(col("has_partial_refund") == 1, 0.5)
    .otherwise(0.0)
)


In [ ]:
# ============================================================
# 8. Build quality_score based on your priority:
#    1) amenities_score
#    2) propery_rating_reviews_score
#    3) host_rating_reviews_score
#    4) cancellation_score
#
#    Weights sum to 1:
#      amenities_score              → 0.35
#      propery_rating_reviews_score → 0.35
#      host_rating_reviews_score    → 0.2
#      cancellation_score           → 0.1
# ============================================================
airbnb_df = airbnb_df.withColumn(
    "quality_score",
    0.35 * col("amenities_score") +
    0.35 * col("propery_rating_reviews_score") + 
    0.2 * col("host_rating_reviews_score") +
    0.1 * col("cancellation_score")
)

# clip to [0, 1] if needed 
airbnb_df = airbnb_df.withColumn(
    "quality_score",
    when(col("quality_score") < 0, 0.0)
    .when(col("quality_score") > 1, 1.0)
    .otherwise(col("quality_score"))
)

# ============================================================
# 9. Superhost boost:
#    - is_supperhost is treated as 0/1
#    - We apply up to +10% boost for superhosts
# ============================================================
airbnb_df = airbnb_df.withColumn(
    "is_supperhost_int",
    when(
        (col("is_supperhost") == "TRUE"),
        lit(1)
    ).otherwise(lit(0))
)

superhost_boost = 0.10  # 10% bonus

airbnb_df = airbnb_df.withColumn(
    "superhost_factor",
    1.0 + superhost_boost * col("is_supperhost_int") 
)

# ============================================================
# 10. Final Value for Money Index
#
#     value_for_money_index_raw =
#         quality_score * price_score * superhost_factor 
#
#     Then clamp to [0, 1] for interpretability.
# ============================================================
airbnb_df = airbnb_df.withColumn(
    "value_for_money",
    col("quality_score") * col("price_score") * col("superhost_factor") 
)

display(airbnb_df)

In [ ]:
# Final Result - Group by Propert_id and return Top 10   

df_property = (
    airbnb_df.groupBy("property_id")
        .agg(
            F.mean("value_for_money").alias("value_for_money"),
        )
)
df_property = df_property.join(
    airbnb_df.select("property_id", "name", "location").distinct(), 
    on="property_id",
    how="left"
)
display(df_property.orderBy(col("value_for_money").desc()).limit(10)) 


### Question 2

#### Load Booking.com data

In [ ]:
train_path = "/dbfs/mnt/lab94290/booking/hw2_train"
train_df = spark.read.parquet(train_path)

test_path = "/dbfs/mnt/lab94290/booking/hw2_test"
test_df = spark.read.parquet(test_path)

In [ ]:
# הצגת הנתונים (בסביבת Databricks מומלץ להשתמש ב-display)
display(test_df)
test_df.printSchema()

display(train_df)
train_df.printSchema()
# אופציה חלופית (רגילה של PySpark)
# test_df.show(5)

# בדיקת הסכמה (שמות העמודות וסוגי המשתנים)

print(train_df.columns)
print(len(train_df.columns))
print(test_df.columns)
print(len(test_df.columns))



#### Add Categorical Mapping to train data

In [ ]:
def rating_to_category(r):    
    if r < 6:
        return "Poor"
    elif r < 7:
        return "Passable"
    elif r < 8:
        return "Good"
    elif r < 9:
        return "Great"
    elif r < 9.5:
        return "Excellent"
    else:
        return "Exceptional"

In [ ]:
from pyspark.sql.types import StringType
from pyspark.sql import functions as F

map_udf = F.udf(rating_to_category, StringType())
train_df = train_df.withColumn("rating_category", map_udf(F.col("review_score")))

#### Your solution

In [ ]:
from pyspark.sql import functions as F

# ----- ממוצע manager_score -----
avg_mgr = train_df.select(F.avg("manager_score")).first()[0]

# ----- ביטוי SQL לציון משוקלל של house_rules -----
house_rules_score_expr = """
aggregate(
  coalesce(house_rules, array()),
  0.0D,
  (acc, x) ->
    acc +
    CASE
      -- Cancellation / prepayment (אנגלית + צרפתית)
      WHEN lower(x.rule) IN (
        'cancellation/ prepayment',
        'annulation / prépaiement'
      ) THEN 4.0

      -- Children & Beds (אנגלית + צרפתית)
      WHEN lower(x.rule) IN (
        'children & beds',
        'children and beds',
        'enfants et lits'
      ) THEN 3.5

      -- Pets (אנגלית + צרפתית)
      WHEN lower(x.rule) IN (
        'pets',
        'animaux domestiques'
      ) THEN 1.75

      -- Check in
      WHEN lower(x.rule) IN (
        'check-in',
        'arrivée'
      ) THEN 2.5

      -- Check out
      WHEN lower(x.rule) IN (
        'check-out',
        'départ'
      ) THEN 2.5

      -- Smoking (אנגלית + צרפתית)
      WHEN lower(x.rule) IN (
        'smoking',
        'fumeurs/non-fumeurs'
      ) THEN 2.5

      -- No age restriction (אנגלית + צרפתית)
      WHEN lower(x.rule) IN (
        'no age restriction',
        'aucune restriction d''âge'
      ) THEN 1.0

      -- Parties
      WHEN lower(x.rule) = 'parties' THEN 3.0

      -- Payments by Booking.com
      WHEN lower(x.rule) = 'payments by booking.com' THEN 0.5

      -- Quiet hours
      WHEN lower(x.rule) = 'quiet hours' THEN 2.5

      -- Age restriction
      WHEN lower(x.rule) = 'age restriction' THEN 2.5

      -- Refundable damage deposit
      WHEN lower(x.rule) = 'refundable damage deposit' THEN 2.25

      -- Cards accepted at this property / hotel
      WHEN lower(x.rule) IN (
        'cards accepted at this property',
        'cards accepted at this hotel'
      ) THEN 0.5

      -- Accepted payment methods
      WHEN lower(x.rule) = 'accepted payment methods' THEN 0.5

      -- Damage policy
      WHEN lower(x.rule) = 'damage policy' THEN 2.0

      -- Cash only
      WHEN lower(x.rule) = 'cash only' THEN 2.5

      -- Groups
      WHEN lower(x.rule) = 'groups' THEN 2.0

      -- Curfew
      WHEN lower(x.rule) = 'curfew' THEN 2.5

      -- כל חוק אחר שלא נכנס לאף קטגוריה
      ELSE 1.0
    END
)
"""

# ----- בניית train_df_filtered עם כל הפיצ'רים -----
train_df_filtered = (
    train_df

    # ---------- House rules: ציון משוקלל ----------
    .withColumn(
        "house_rules_score",
        F.expr(house_rules_score_expr)
    )

    # ---------- Languages ----------
    # manager_language_spoken : ARRAY<STRING>
    .withColumn(
        "languages",
        F.coalesce(F.col("manager_language_spoken"), F.array())
    )

    # האם יש English
    .withColumn(
        "has_english",
        F.when(
            F.array_contains(F.col("languages"), "English"),
            1
        ).otherwise(0)
    )

    # כמה שפות יש
    .withColumn(
        "num_languages",
        F.size(F.col("languages"))
    )

    # ---------- Manager Score ----------
    .withColumn(
        "manager_score",
        F.when(
            F.col("manager_score").isNull(),
            F.lit(avg_mgr)
        ).otherwise(F.col("manager_score"))
    )

    # ---------- Metro ----------
    .withColumn(
        "metro_access",
        F.col("metro_railway_access").cast("int")
    )

    # ---------- DataFrame סופי ----------
    .select(
        "hotel_id",
        "house_rules_score",
        "has_english",
        "num_languages",
        "manager_score",
        "metro_access",
        "review_score"
    )
)


In [ ]:
from pyspark.sql import functions as F

rules_df = (
    train_df
    .select(F.explode(F.coalesce(F.col("house_rules"), F.array())).alias("rule_struct"))
    .select(F.col("rule_struct.rule").alias("rule_name"))
    .where(F.col("rule_name").isNotNull())
)

# רשימת כל סוגי החוקים (distinct)
distinct_rules_df = rules_df.distinct()

distinct_rules_df.show(truncate=False)
print("Total distinct rule types:", distinct_rules_df.count())

rules_counts_df = (
    rules_df
    .groupBy("rule_name")
    .count()
    .orderBy(F.desc("count"))
)

rules_counts_df.show(100, truncate=False)  # תוכל להגביר/להקטין את 100



In [ ]:
import sparknlp
from sparknlp.base import *
from sparknlp.annotator import *
from pyspark.ml import Pipeline
from pyspark.sql.functions import col, size, array_distinct, when, explode, avg, lower, expr

# ============================================================
# 0. Define Amenity Weights & Logic
#    We create a SQL expression string to sum up points dynamically.
# ============================================================
amenity_weights = {
    "pool": 3.0,
    "spa": 3.0,
    "wellness": 3.0,
    "hot tub": 4.0,
    "jacuzzi": 4.0,
    "beach": 4.0,
    "ski": 3.0,
    "breakfast": 2.0,
    "parking": 1.5,
    "wifi": 1.0,
    "air conditioning": 1.5,
    "balcony": 1.5,
    "terrace": 1.5,
    "view": 2.0
}

# Construct the SQL expression:
# "CASE WHEN exists(most_popular_facilities, x -> lower(x) like '%pool%') THEN 3.0 ELSE 0 END + ..."
score_parts = []
for key, weight in amenity_weights.items():
    part = f"CASE WHEN exists(most_popular_facilities, x -> lower(x) like '%{key}%') THEN {weight} ELSE 0.0 END"
    score_parts.append(part)

final_score_expr = " + ".join(score_parts)


# ==========================================
# STEP 1: Feature Engineering (Counts & Scores)
# ==========================================
features_df = train_df.select(
    col("hotel_id"),
    col("number_of_reviews"),

    # 1. Simple Count (Legacy)
    when(col("most_popular_facilities").isNotNull(), 
         size(array_distinct(col("most_popular_facilities"))))
    .otherwise(0).alias("most_popular_facilities_count"),
    
    # 2. NEW: Weighted Amenities Score
    #    We apply the expression we built above.
    when(col("most_popular_facilities").isNotNull(), 
         expr(final_score_expr))
    .otherwise(0.0).alias("amenities_weighted_score"),
    
    # 3. Count surroundings
    when(col("property_surroundings").isNotNull(), 
         size(col("property_surroundings")))
    .otherwise(0).alias("property_surroundings_count"),
    
    # Keep top_reviews for the NLP step
    col("top_reviews")
)

# ==========================================
# STEP 2: Spark NLP Pipeline
# ==========================================
# A. Explode reviews to get one row per review text
reviews_flat_df = features_df.select(
    col("hotel_id"),
    explode(col("top_reviews")).alias("review_struct")
).select(
    col("hotel_id"),
    col("review_struct.review").alias("text")
).filter(col("text").isNotNull() & (col("text") != ""))

# B. Build Pipeline
document_assembler = DocumentAssembler()\
    .setInputCol("text")\
    .setOutputCol("document")

use = UniversalSentenceEncoder.pretrained(name="tfhub_use", lang="en") \
    .setInputCols(["document"])\
    .setOutputCol("sentence_embeddings")

sentiment_dl = SentimentDLModel.pretrained(name="sentimentdl_use_twitter", lang="en") \
    .setInputCols(["sentence_embeddings"])\
    .setOutputCol("sentiment")

pipeline = Pipeline(stages=[document_assembler, use, sentiment_dl])

# C. Run the model
print("Running NLP Pipeline on full dataset...")
model = pipeline.fit(reviews_flat_df)
processed_reviews_df = model.transform(reviews_flat_df)

# ==========================================
# STEP 3: Validated Scoring Logic
# ==========================================
scores_df = processed_reviews_df.select(
    col("hotel_id"),
    lower(col("sentiment.result").getItem(0)).alias("label"),
    col("sentiment.metadata").getItem(0).getItem("positive").cast("float").alias("pos_score"),
    col("sentiment.metadata").getItem(0).getItem("negative").cast("float").alias("neg_score")
).withColumn(
    "review_score_continuous",
    when(col("label") == "positive", col("pos_score"))
    .when(col("label") == "negative", -col("neg_score"))
    .otherwise(0.0)
)

# Aggregate back to Hotel Level
hotel_sentiment_df = scores_df.groupBy("hotel_id").agg(
    avg("review_score_continuous").alias("avg_sentiment_score")
)

# ==========================================
# STEP 4: Final Merge & Display
# ==========================================
final_df = features_df.drop("top_reviews").join(hotel_sentiment_df, on="hotel_id", how="left") \
    .na.fill({"avg_sentiment_score": 0.0})

# Select the final columns, including the new weighted score
final_output = final_df.select(
    "hotel_id",
    "number_of_reviews",
    "most_popular_facilities_count",
    "amenities_weighted_score",     # <--- The new smart feature
    "property_surroundings_count",
    "avg_sentiment_score"
)


In [ ]:
full_train_df = (
    train_df_filtered
    .join(
        final_output,
        on="hotel_id",
        how="left"
    )
)

In [ ]:
from pyspark.ml.classification import RandomForestClassifier
from pyspark.ml import Pipeline
from pyspark.ml.evaluation import MulticlassClassificationEvaluator
from pyspark.ml.tuning import TrainValidationSplit, ParamGridBuilder
from pyspark.sql.functions import col, log1p, lit, size, when, array_contains
from pyspark.ml.feature import Bucketizer, Imputer, VectorAssembler, IndexToString


# --- 1. DATA PREPARATION (Training) ---

# A. Create the Log Feature & Clean Data
# We apply log1p (Log(x+1)) because log(0) is undefined.
print("Preparing training data...")
train_prep = full_train_df.withColumn("log_reviews", log1p(col("number_of_reviews")))

# B. Remove rows without a target
train_prep = train_prep.dropna(subset=["review_score"])

# C. Sample & Cache
# We use 80% of data. Since we dropped bad features, we can use more data than before.
train_df = train_prep.sample(False, 0.8, seed=42)
train_df.cache()

print(f"Training on {train_df.count()} rows (Log-transformed & Optimized)")

# --- 2. SETUP PIPELINE ---

# Target Labels (Same as PDF)
bucketizer = Bucketizer(
    splits=[0, 6, 7, 8, 9, 9.5, float('inf')],
    inputCol="review_score",
    outputCol="label",
    handleInvalid="skip"
)

label_list = ["Poor", "Passable", "Good", "Great", "Excellent", "Exceptional"]

# OPTIMIZED FEATURE LIST 
# Removed: has_english, num_languages, metro_access, property_surroundings_count
# Added: log_reviews (Replacing number_of_reviews)
feature_cols = [
    "house_rules_score", 
    "manager_score", 
    "log_reviews",                  # <--- The new Log feature
    "most_popular_facilities_count", 
    "amenities_weighted_score", 
    "avg_sentiment_score"
]

imputer = Imputer(
    inputCols=feature_cols,
    outputCols=[f"{c}_imputed" for c in feature_cols]
).setStrategy("mean")

assembler = VectorAssembler(
    inputCols=[f"{c}_imputed" for c in feature_cols],
    outputCol="features",
    handleInvalid="keep"
)

# STRONGER MODEL
rf = RandomForestClassifier(
    labelCol="label",
    featuresCol="features",
    seed=42,
    numTrees=60,   # Increased from 20 (Better accuracy)
    maxDepth=8     # Increased from 5 (Learns more complex patterns)
)

label_converter = IndexToString(
    inputCol="prediction",
    outputCol="rating_category",
    labels=label_list
)

pipeline = Pipeline(stages=[bucketizer, imputer, assembler, rf, label_converter])

# --- 3. TRAINING ---

# We only check one grid point to save time, but it's a "stronger" point.
paramGrid = (ParamGridBuilder()
    .addGrid(rf.numTrees, [60])
    .addGrid(rf.maxDepth, [8]) 
    .build()
)

evaluator = MulticlassClassificationEvaluator(
    labelCol="label",
    predictionCol="prediction",
    metricName="weightedFMeasure"
)

# 90/10 Split per assignment requirement
tvs = TrainValidationSplit(
    estimator=pipeline,
    estimatorParamMaps=paramGrid,
    evaluator=evaluator,
    trainRatio=0.9, 
    seed=42
)

print("Starting training with optimized features...")
model = tvs.fit(train_df)
print("-" * 30)
print(f"Validation F1 Score: {model.validationMetrics[0]}")
print("-" * 30)



In [ ]:
import pandas as pd

# 1. Access the best Pipeline from your Tuning results
best_pipeline = model.bestModel

# 2. Extract the Random Forest stage
# Your pipeline structure is: [bucketizer, imputer, assembler, rf, label_converter]
# So RF is at index -2 (second to last)
rf_model = best_pipeline.stages[-2]

# 3. Get the raw importance scores
importances = rf_model.featureImportances

# 4. Define the feature names (Must match the list you used for training!)
# Based on the optimized code I sent you, this should be the list:
feature_cols = [
    "house_rules_score", 
    "manager_score", 
    "log_reviews", 
    "most_popular_facilities_count", 
    "amenities_weighted_score", 
    "avg_sentiment_score"
]

# 5. Combine and Sort
importance_list = []
for i in range(len(feature_cols)):
    # usage of float() ensures it prints nicely
    importance_list.append((feature_cols[i], float(importances[i])))

# Create a Pandas DataFrame for clean display
df_imp = pd.DataFrame(importance_list, columns=["Feature", "Importance"])
df_imp = df_imp.sort_values(by="Importance", ascending=False)

print("-" * 30)
print(f"Model F1 Score: 0.436")
print("-" * 30)
print("Most Important Features:")
print(df_imp)

In [ ]:
from pyspark.sql import functions as F
from pyspark.ml import Pipeline, PipelineModel
from pyspark.ml.feature import Bucketizer, Imputer, VectorAssembler
from pyspark.ml.classification import RandomForestClassifier
from pyspark.ml.evaluation import MulticlassClassificationEvaluator

# ============================================================
# Re-train the model on ALL training data (full_train_df)
# ============================================================

# לוג על number_of_reviews גם בטריין המלא
full_train_prep = full_train_df.withColumn(
    "log_reviews", F.log1p(F.col("number_of_reviews"))
).dropna(subset=["review_score"])

print(f"Full train rows: {full_train_prep.count()}")

# נשתמש באותם היפר-פרמטרים שה-TVSplit מצא
best_rf_model = model.bestModel.stages[-2]  # ה-RF מהמודל הטוב ביותר
best_num_trees = best_rf_model.getNumTrees
best_max_depth = best_rf_model.getOrDefault("maxDepth")

print(f"Using best params: numTrees={best_num_trees}, maxDepth={best_max_depth}")

# 1.1 Bucketizer (אותו כמו קודם)
bucketizer_full = Bucketizer(
    splits=[0, 6, 7, 8, 9, 9.5, float('inf')],
    inputCol="review_score",
    outputCol="label",
    handleInvalid="skip"
)

# 1.2 Features list (אותו כמו קודם)
feature_cols = [
    "house_rules_score", 
    "manager_score", 
    "log_reviews", 
    "most_popular_facilities_count", 
    "amenities_weighted_score", 
    "avg_sentiment_score"
]

imputer_full = Imputer(
    inputCols=feature_cols,
    outputCols=[f"{c}_imputed" for c in feature_cols]
).setStrategy("mean")

assembler_full = VectorAssembler(
    inputCols=[f"{c}_imputed" for c in feature_cols],
    outputCol="features",
    handleInvalid="keep"
)

rf_full = RandomForestClassifier(
    labelCol="label",
    featuresCol="features",
    seed=42,
    numTrees=best_num_trees,
    maxDepth=best_max_depth
)

label_list = ["Poor", "Passable", "Good", "Great", "Excellent", "Exceptional"]

label_converter_full = IndexToString(
    inputCol="prediction",
    outputCol="rating_category",
    labels=label_list
)

pipeline_full = Pipeline(stages=[bucketizer_full, imputer_full, assembler_full, rf_full, label_converter_full])

print("Fitting final model on ALL training data...")
final_model = pipeline_full.fit(full_train_prep)
print("Done training final model.")

# (אופציונלי) לבדוק F1 על הטריין המלא
train_pred = final_model.transform(full_train_prep)
train_evaluator = MulticlassClassificationEvaluator(
    labelCol="label",
    predictionCol="prediction",
    metricName="weightedFMeasure"
)
train_f1 = train_evaluator.evaluate(train_pred)
print(f"Train weighted F1 on full data: {train_f1:.3f}")


# ============================================================
# להכין פיצ'רים ל-test_df
# ============================================================

# 2.1 house_rules, languages, manager_score, metro (בדומה ל-train_df_filtered)
test_df_filtered = (
    test_df
    .withColumn(
        "house_rules_score",
        F.expr(house_rules_score_expr)
    )
    .withColumn(
        "languages",
        F.coalesce(F.col("manager_language_spoken"), F.array())
    )
    .withColumn(
        "has_english",
        F.when(F.array_contains(F.col("languages"), "English"), 1).otherwise(0)
    )
    .withColumn(
        "num_languages",
        F.size(F.col("languages"))
    )
    .withColumn(
        "manager_score",
        F.when(F.col("manager_score").isNull(), F.lit(avg_mgr)).otherwise(F.col("manager_score"))
    )
    .withColumn(
        "metro_access",
        F.col("metro_railway_access").cast("int")
    )
    .select(
        "hotel_id",
        "house_rules_score",
        "has_english",
        "num_languages",
        "manager_score",
        "metro_access"
    )
)

# 2.2 most_popular_facilities + surroundings + top_reviews (בדומה ל-features_df)
test_features_df = test_df.select(
    F.col("hotel_id"),
    F.col("number_of_reviews"),
    F.when(
        F.col("most_popular_facilities").isNotNull(),
        F.size(F.array_distinct(F.col("most_popular_facilities")))
    ).otherwise(0).alias("most_popular_facilities_count"),
    F.when(
        F.col("most_popular_facilities").isNotNull(),
        F.expr(final_score_expr)
    ).otherwise(0.0).alias("amenities_weighted_score"),
    F.when(
        F.col("property_surroundings").isNotNull(),
        F.size(F.col("property_surroundings"))
    ).otherwise(0).alias("property_surroundings_count"),
    F.col("top_reviews")
)

# 2.3 NLP על הטקסטים ב-test (בדומה ל-reviews_flat_df וכו')
test_reviews_flat_df = test_features_df.select(
    F.col("hotel_id"),
    F.explode(F.col("top_reviews")).alias("review_struct")
).select(
    F.col("hotel_id"),
    F.col("review_struct.review").alias("text")
).filter(F.col("text").isNotNull() & (F.col("text") != ""))

# לבנות מחדש את ה-pipeline של ה-NLP (אותו כמו קודם)
document_assembler_test = DocumentAssembler()\
    .setInputCol("text")\
    .setOutputCol("document")

use_test = UniversalSentenceEncoder.pretrained(name="tfhub_use", lang="en") \
    .setInputCols(["document"])\
    .setOutputCol("sentence_embeddings")

sentiment_dl_test = SentimentDLModel.pretrained(name="sentimentdl_use_twitter", lang="en") \
    .setInputCols(["sentence_embeddings"])\
    .setOutputCol("sentiment")

sentiment_pipeline_test = Pipeline(stages=[document_assembler_test, use_test, sentiment_dl_test])

print("Running NLP pipeline on test reviews...")
sentiment_model_test = sentiment_pipeline_test.fit(test_reviews_flat_df)
test_processed_reviews_df = sentiment_model_test.transform(test_reviews_flat_df)

test_scores_df = test_processed_reviews_df.select(
    F.col("hotel_id"),
    F.lower(F.col("sentiment.result").getItem(0)).alias("label"),
    F.col("sentiment.metadata").getItem(0).getItem("positive").cast("float").alias("pos_score"),
    F.col("sentiment.metadata").getItem(0).getItem("negative").cast("float").alias("neg_score")
).withColumn(
    "review_score_continuous",
    F.when(F.col("label") == "positive", F.col("pos_score"))
     .when(F.col("label") == "negative", -F.col("neg_score"))
     .otherwise(0.0)
)

test_hotel_sentiment_df = test_scores_df.groupBy("hotel_id").agg(
    F.avg("review_score_continuous").alias("avg_sentiment_score")
)

# 2.4 איחוד פיצ'רים ל-full_test_df (בדומה ל-final_output/full_train_df)
test_final_output = (
    test_features_df
    .drop("top_reviews")
    .join(test_hotel_sentiment_df, on="hotel_id", how="left")
    .na.fill({"avg_sentiment_score": 0.0})
)

full_test_df = (
    test_df_filtered
    .join(test_final_output, on="hotel_id", how="left")
)

# להוסיף log_reviews גם לטסט
test_prep = full_test_df.withColumn(
    "log_reviews", F.log1p(F.col("number_of_reviews"))
)

# ============================================================
# שימוש במודל המאומן כדי לנבא על ה-test
# ============================================================

# שלבים פיטדיים מה-final_model
bucketizer_model = final_model.stages[0]
imputer_model    = final_model.stages[1]
assembler_model  = final_model.stages[2]
rf_model_full    = final_model.stages[3]
label_conv_model = final_model.stages[4]

# PipelineModel חדש רק לפיצ'רים + RF + המרת label למילים
predict_model = PipelineModel(stages=[
    imputer_model,
    assembler_model,
    rf_model_full,
    label_conv_model
])

print("Predicting on test set...")
test_predictions = predict_model.transform(test_prep)

# ניקח רק מה שצריך להגשה
submission_df = test_predictions.select("hotel_id", "rating_category")

display(submission_df)


In [ ]:
# ============================================================
# SAVE & VALIDATE SUBMISSION FILE
# ============================================================

# Path required by assignment format (replace IDs if needed)
output_path = "/mnt/lab94290/results/test_000000000_000000000_000000000.csv"

# Save predictions
submission_df.write.mode("overwrite").csv(output_path, header=True)

print("✅ File saved to:", output_path)

# ============================================================
# VALIDATE OUTPUT FILE
# ============================================================

df = spark.read.csv(output_path, header=True)

print("✅ Validation – preview of saved file:")
display(df)

print("✅ Schema:")
df.printSchema()

print("✅ Row count:")
print(df.count())


________________________